# Concept-Aware Training — Research Tasks 5–8

**Second round of experiments June 22.**

Tasks 1–4 showed that the contrastive model achieves the best accuracy among all concept-trained variants (+6pp over original NCP), while all concept-trained models underperform the CLM baseline on NTP PPL. Priority order:

| Task | Experiment | Purpose |
|------|-----------|----------|
| **5** | Dual Evaluation (Concept PPL + NTP PPL) | Report both metrics side-by-side as requested |
| **6** | Syn-Only Fair Comparison | Control for data size/domain before attributing gain to objective |
| **7** | Downstream Eval: SNLI + SPAM | Highest priority — does the gain transfer to NLU tasks? |
| **8** | Hard-Negative Strategy Ablation | Which WordNet strategy drives the contrastive gain? |

**Before running:** `Runtime > Change runtime type > T4 GPU`

---
## 0. Setup

In [ ]:
import subprocess, torch
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,memory.free', '--format=csv,noheader'],
                        capture_output=True, text=True)
print(result.stdout)
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None"}')

In [ ]:
!pip install -q "transformers>=4.41.0,<5.0.0" datasets accelerate evaluate nltk

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

REPO_URL = 'https://github.com/SharvaGogawale1/concept-aware-training.git'
REPO_DIR = '/content/concept_aware_training'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull origin main

%cd {REPO_DIR}
print('Repo ready.')

In [ ]:
import os, json
import pandas as pd

# ── Paths (match conventions from main notebook) ──────────────────────────────
DRIVE_DATASET_ROOT = '/content/concept_aware_training/data/'
OUTPUT_ROOT        = '/content/drive/MyDrive/concept_aware_outputs'   # <-- UPDATE if needed
MODEL_LOCAL_PATH   = '/content/Llama-3.2-1B'                          # must be downloaded
SCRIPTS_DIR        = '/content/concept_aware_training/transformers/examples/pytorch/language-modeling'

# ── Existing checkpoints from Tasks 1–4 ──────────────────────────────────────
CLM_CKPT         = os.path.join(OUTPUT_ROOT, 'standard_syn')
SYN_NCP_CKPT     = os.path.join(OUTPUT_ROOT, 'syn_custom_loss')
HYP_NCP_CKPT     = os.path.join(OUTPUT_ROOT, 'hyp_custom_loss')
DIFF_NCP_CKPT    = os.path.join(OUTPUT_ROOT, 'diff_ncp_syn')
CONTRASTIVE_CKPT = os.path.join(OUTPUT_ROOT, 'contrastive')

ALL_CKPTS  = [CLM_CKPT, SYN_NCP_CKPT, HYP_NCP_CKPT, DIFF_NCP_CKPT, CONTRASTIVE_CKPT]
CKPT_LABELS = ['CLM', 'Syn-NCP', 'Hyp-NCP', 'Diff-NCP', 'Contrastive']

# ── Data paths ────────────────────────────────────────────────────────────────
SYN_CONCEPT_TRAIN = os.path.join(DRIVE_DATASET_ROOT, 'syn/youtube/context_loss_train.csv')
SYN_CONCEPT_VAL   = os.path.join(DRIVE_DATASET_ROOT, 'syn/youtube/context_loss_val.csv')
HYP_CONCEPT_TRAIN = os.path.join(DRIVE_DATASET_ROOT, 'hyp/youtube/context_loss_train.csv')
HYP_CONCEPT_VAL   = os.path.join(DRIVE_DATASET_ROOT, 'hyp/youtube/context_loss_val.csv')
VANILLA_VAL       = os.path.join(DRIVE_DATASET_ROOT, 'hyp/youtube/vanilla_val.txt')

# Verify key paths exist
for label, path in [('SYN_CONCEPT_VAL', SYN_CONCEPT_VAL),
                     ('HYP_CONCEPT_VAL', HYP_CONCEPT_VAL),
                     ('VANILLA_VAL',     VANILLA_VAL)]:
    status = '✅' if os.path.exists(path) else '❌ NOT FOUND'
    print(f'{status}  {label}: {path}')

print()
for label, path in zip(CKPT_LABELS, ALL_CKPTS):
    status = '✅' if os.path.exists(path) else '❌ NOT FOUND'
    print(f'{status}  {label}: {path}')

---
## Task 5 — Dual Evaluation: Concept PPL + NTP PPL

Check both metrics reported together. `eval_concept_ppl.py` computes them in one shot:

- **NTP PPL / Acc** — standard CLM loss on `vanilla_val.txt` (same as Task 1)
- **Set-Marginal Concept PPL** — `exp(mean(-log Σ_{c ∈ C} p(c|ctx)))` on `context_loss_val.csv`

The set-marginal PPL is the cleanest intrinsic measure of concept coverage: it asks "does the model assign meaningful total probability mass to the full valid synonym/hypernym set?", regardless of which gold surface form appeared in training.

**Expected direction**: NCP and contrastive models should have lower concept PPL than CLM (they were trained to improve it), even as their NTP PPL is higher.

In [ ]:
# Task 5a: Dual eval using synonym concept validation set

DUAL_RESULTS_SYN = '/content/dual_eval_syn_results.json'

existing_ckpts = [p for p in ALL_CKPTS if os.path.exists(p)]
ckpt_args = ' '.join(existing_ckpts)
print(f'Evaluating {len(existing_ckpts)} checkpoints on SYN concept val ...')

%cd {SCRIPTS_DIR}
!python eval_concept_ppl.py \
  --checkpoints {ckpt_args} \
  --concept_csv {SYN_CONCEPT_VAL} \
  --vanilla_val {VANILLA_VAL} \
  --results_json {DUAL_RESULTS_SYN}

In [ ]:
# Task 5b: Dual eval using hypernym concept validation set

DUAL_RESULTS_HYP = '/content/dual_eval_hyp_results.json'

%cd {SCRIPTS_DIR}
!python eval_concept_ppl.py \
  --checkpoints {ckpt_args} \
  --concept_csv {HYP_CONCEPT_VAL} \
  --vanilla_val {VANILLA_VAL} \
  --results_json {DUAL_RESULTS_HYP}

In [ ]:
# Task 5 results: display side-by-side comparison table

import json, pandas as pd, os

def load_dual_results(path, concept_type):
    if not os.path.exists(path):
        print(f'  Results file not found: {path}')
        return pd.DataFrame()
    results = json.load(open(path))
    rows = []
    for r in results:
        if 'error' in r:
            continue
        rows.append({
            'Model':          os.path.basename(r['checkpoint'].rstrip('/')),
            'NTP PPL':        r.get('ntp_ppl', float('nan')),
            'NTP Acc':        f"{r.get('ntp_acc', float('nan')):.3f}",
            f'{concept_type} Concept PPL': r.get('concept_ppl', float('nan')),
            'Coverage':       f"{r.get('concept_coverage_pct', 0):.1f}%",
            '#Slots':         r.get('concept_n_evaluated', 0),
        })
    return pd.DataFrame(rows)

df_syn = load_dual_results(DUAL_RESULTS_SYN, 'Syn')
df_hyp = load_dual_results(DUAL_RESULTS_HYP, 'Hyp')

if not df_syn.empty:
    print('=== Synonym Concept Eval ===')
    display(df_syn)

if not df_hyp.empty:
    print('\n=== Hypernym Concept Eval ===')
    display(df_hyp)

In [ ]:
# Save dual eval results to Drive
import shutil
os.makedirs(OUTPUT_ROOT, exist_ok=True)
for src, name in [(DUAL_RESULTS_SYN, 'dual_eval_syn_results.json'),
                   (DUAL_RESULTS_HYP, 'dual_eval_hyp_results.json')]:
    if os.path.exists(src):
        dst = os.path.join(OUTPUT_ROOT, name)
        shutil.copy(src, dst)
        print(f'Saved: {dst}')

---
## Task 6 — Syn-Only Fair Comparison

**Why this is critical before Task 7**: The contrastive model trained on merged syn+hyp data (2,241 examples) is being compared against NCP models trained on syn-only (~1,828 examples). This confounds *objective* with *data size and domain mixture*.

This task trains a contrastive model on syn-only data to isolate the effect of the objective:

- **If syn-only contrastive > syn NCP** → the gain is from the InfoNCE objective itself ✓
- **If syn-only contrastive ≈ syn NCP** → the earlier gain was from data size/domain mixing, not the objective

Steps: (1) build syn-only contrastive CSV, (2) train, (3) dual eval.

In [ ]:
# Task 6 Step 1: Ensure WordNet is available
import nltk
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)
print('WordNet ready.')

In [ ]:
# Task 6 Step 2: Build syn-only contrastive dataset
# Uses the same build script as Task 3 but restricts to synonym source only.

SYN_ONLY_CONTRASTIVE_DIR = os.path.join(DRIVE_DATASET_ROOT, 'contrastive/youtube_syn_only')

%cd /content/concept_aware_training
!python build_contrastive_dataset.py \
  --source syn \
  --max_negatives 10 \
  --output_dir {SYN_ONLY_CONTRASTIVE_DIR}

SYN_ONLY_CONTRASTIVE_TRAIN = os.path.join(SYN_ONLY_CONTRASTIVE_DIR, 'syn_contrastive_train.csv')
SYN_ONLY_CONTRASTIVE_VAL   = os.path.join(SYN_ONLY_CONTRASTIVE_DIR, 'syn_contrastive_val.csv')

if os.path.exists(SYN_ONLY_CONTRASTIVE_TRAIN):
    df = pd.read_csv(SYN_ONLY_CONTRASTIVE_TRAIN)
    print(f'\nSyn-only contrastive train: {len(df)} rows')
    print(df.head(2))

In [ ]:
# Task 6 Step 3: Train contrastive model on syn-only data
# Same hyperparameters as Task 4 contrastive run — only the training data differs.

SYN_ONLY_CONTRASTIVE_OUT = os.path.join(OUTPUT_ROOT, 'contrastive_syn_only')

%cd {SCRIPTS_DIR}
!python run_clm_contrastive.py \
  --model_name_or_path {MODEL_LOCAL_PATH} \
  --train_file {SYN_ONLY_CONTRASTIVE_TRAIN} \
  --validation_file {SYN_ONLY_CONTRASTIVE_VAL} \
  --ncp_alpha 0.5 \
  --contrast_beta 1.0 \
  --block_size 128 \
  --torch_dtype bfloat16 \
  --bf16 True \
  --gradient_accumulation_steps 8 \
  --per_device_train_batch_size 2 \
  --per_device_eval_batch_size 2 \
  --auto_find_batch_size True \
  --overwrite_output_dir \
  --do_train \
  --do_eval \
  --output_dir {SYN_ONLY_CONTRASTIVE_OUT}

In [ ]:
# Task 6 Step 4: Dual eval — compare syn-only contrastive against the merged contrastive
# and original syn NCP (the three-way comparison that answers the confound question).

TASK6_RESULTS = '/content/task6_syn_only_comparison.json'

comparison_ckpts = [
    CLM_CKPT,
    SYN_NCP_CKPT,
    CONTRASTIVE_CKPT,           # merged syn+hyp contrastive (Task 4)
    SYN_ONLY_CONTRASTIVE_OUT,   # syn-only contrastive (Task 6)
]
comparison_ckpts = [p for p in comparison_ckpts if os.path.exists(p)]
ckpt_args = ' '.join(comparison_ckpts)

%cd {SCRIPTS_DIR}
!python eval_concept_ppl.py \
  --checkpoints {ckpt_args} \
  --concept_csv {SYN_CONCEPT_VAL} \
  --vanilla_val {VANILLA_VAL} \
  --results_json {TASK6_RESULTS}

In [ ]:
# Task 6 results: the critical comparison
if os.path.exists(TASK6_RESULTS):
    results = json.load(open(TASK6_RESULTS))
    rows = []
    labels = {'standard_syn': 'CLM (baseline)',
              'syn_custom_loss': 'Syn-NCP (original)',
              'contrastive': 'Contrastive — merged syn+hyp (Task 4)',
              'contrastive_syn_only': 'Contrastive — syn-only (Task 6)'}
    for r in results:
        if 'error' in r:
            continue
        key = os.path.basename(r['checkpoint'].rstrip('/'))
        rows.append({
            'Model': labels.get(key, key),
            'NTP PPL': r.get('ntp_ppl'),
            'NTP Acc': f"{r.get('ntp_acc', 0):.3f}",
            'Concept PPL (Syn)': r.get('concept_ppl'),
            'Coverage': f"{r.get('concept_coverage_pct', 0):.1f}%",
        })
    df6 = pd.DataFrame(rows)
    print('=== Task 6: Syn-Only Fair Comparison ===')
    display(df6)
    shutil.copy(TASK6_RESULTS, os.path.join(OUTPUT_ROOT, 'task6_syn_only_comparison.json'))
else:
    print('Results not yet available — run the eval cell above first.')

---
## Task 7 — Downstream Evaluation: SNLI + SPAM

**Highest priority. ** Tests whether the concept-accuracy gain transfers to downstream NLU.

**SNLI** (`stanfordnlp/snli`, 3-class NLI: entailment / neutral / contradiction).  
Hypothesis: hypernym-aware training should improve entailment detection, since understanding that *cat → animal* is exactly the is-a relationship NLI tests.

**SPAM** (`talby/spamassassin`, config=`"text"` — SpamAssassin public mail corpus, binary ham/spam).  
Dataset has only a `train` split (10.7k rows); the script auto-creates an 80/20 train/val split.

**Protocol**:
1. **Linear probe first** (`--freeze_base`): freeze LM weights, train only the classification head. Isolates pre-training representation quality from fine-tuning capacity.
2. **Full fine-tune**: train all parameters. Best downstream accuracy, standard comparison.


In [ ]:
# Task 7a: SNLI — Linear probe (freeze_base)
# Trains only the classification head. Cleanest measure of pre-training quality.

DOWNSTREAM_DIR   = os.path.join(OUTPUT_ROOT, 'downstream')
SNLI_PROBE_JSON  = '/content/snli_probe_results.json'

ckpt_args = ' '.join([p for p in ALL_CKPTS if os.path.exists(p)])

%cd {SCRIPTS_DIR}
!python run_downstream_eval.py \
  --checkpoints {ckpt_args} \
  --task snli \
  --freeze_base \
  --max_train_samples 20000 \
  --num_epochs 5 \
  --output_dir {DOWNSTREAM_DIR} \
  --results_json {SNLI_PROBE_JSON}

In [ ]:
# Task 7b: SNLI — Full fine-tune
# All parameters trained. Measures best achievable downstream performance.

SNLI_FINETUNE_JSON = '/content/snli_finetune_results.json'

%cd {SCRIPTS_DIR}
!python run_downstream_eval.py \
  --checkpoints {ckpt_args} \
  --task snli \
  --max_train_samples 20000 \
  --num_epochs 3 \
  --output_dir {DOWNSTREAM_DIR} \
  --results_json {SNLI_FINETUNE_JSON}

In [ ]:
# Task 7c: SPAM (SpamAssassin talby/spamassassin 'text') -- Linear probe
# Auto-splits train 80/20 into train/val inside run_downstream_eval.py

DOWNSTREAM_DIR  = os.path.join(OUTPUT_ROOT, 'downstream')
SPAM_PROBE_JSON = os.path.join(OUTPUT_ROOT, 'spam_probe_results.json')
ckpt_args = ' '.join([p for p in ALL_CKPTS if os.path.exists(p)])

import os as _os
_os.chdir(SCRIPTS_DIR)
_os.system(
    f"python run_downstream_eval.py"
    f" --checkpoints {ckpt_args}"
    f" --task spam"
    f" --freeze_base"
    f" --num_epochs 5"
    f" --output_dir {DOWNSTREAM_DIR}"
    f" --results_json {SPAM_PROBE_JSON}"
)

In [ ]:
# Task 7d: SPAM (SpamAssassin) -- Full fine-tune

DOWNSTREAM_DIR     = os.path.join(OUTPUT_ROOT, 'downstream')
SPAM_FINETUNE_JSON = os.path.join(OUTPUT_ROOT, 'spam_finetune_results.json')
ckpt_args = ' '.join([p for p in ALL_CKPTS if os.path.exists(p)])

import os as _os
_os.chdir(SCRIPTS_DIR)
_os.system(
    f"python run_downstream_eval.py"
    f" --checkpoints {ckpt_args}"
    f" --task spam"
    f" --num_epochs 3"
    f" --output_dir {DOWNSTREAM_DIR}"
    f" --results_json {SPAM_FINETUNE_JSON}"
)

In [ ]:
# Task 7 results: display all downstream comparisons

def load_downstream(path, task_label, mode_label):
    if not os.path.exists(path):
        return pd.DataFrame()
    results = json.load(open(path))
    rows = []
    for r in results:
        if 'error' in r:
            continue
        rows.append({
            'Model':     os.path.basename(r['checkpoint'].rstrip('/')),
            'Task':      task_label,
            'Mode':      mode_label,
            'Accuracy':  r.get('accuracy', float('nan')),
            'F1':        r.get('f1', float('nan')),
        })
    return pd.DataFrame(rows)

frames = [
    load_downstream(SNLI_PROBE_JSON,    'SNLI',  'Linear Probe'),
    load_downstream(SNLI_FINETUNE_JSON, 'SNLI',  'Full Fine-Tune'),
    load_downstream(SPAM_PROBE_JSON,    'SPAM',  'Linear Probe'),
    load_downstream(SPAM_FINETUNE_JSON, 'SPAM',  'Full Fine-Tune'),
]
df7 = pd.concat([f for f in frames if not f.empty], ignore_index=True)

if not df7.empty:
    print('=== Task 7: Downstream Evaluation ===')
    display(df7.pivot_table(index='Model', columns=['Task','Mode'],
                            values=['Accuracy','F1'], aggfunc='first').round(4))
    # Save to Drive
    for src, name in [(SNLI_PROBE_JSON,    'snli_probe_results.json'),
                       (SNLI_FINETUNE_JSON, 'snli_finetune_results.json'),
                       (SPAM_PROBE_JSON,    'spam_probe_results.json'),
                       (SPAM_FINETUNE_JSON, 'spam_finetune_results.json')]:
        if os.path.exists(src):
            shutil.copy(src, os.path.join(OUTPUT_ROOT, name))
else:
    print('No downstream results yet — run the cells above first.')

---
## Task 8 — Hard-Negative Strategy Ablation

If the contrastive approach holds up after Tasks 6–7, this ablation identifies which WordNet strategy is responsible for the gain.

Three ablation models, each trained with only one negative type:

| Strategy | Description | Notes |
|---|---|---|
| `co_hyponym` | Siblings in the WordNet hierarchy (same parent) | May still be valid in context — be cautious |
| `wrong_sense` | Other senses of the same word (e.g., river bank vs. financial bank) | **Recommended safer option** |
| `same_pos` | Random same-POS WordNet words | Weakest signal, least structured |

Compare: Full (all three) > Co-hyponym only > Wrong-sense only > Same-POS only

The strategy that, when ablated away, causes the biggest drop explains the most of the gain.

In [ ]:
# Task 8 Step 1: Build one ablation dataset per strategy
# Each uses --source both (merged syn+hyp) and a single --strategy.

ABLATION_STRATEGIES = ['co_hyponym', 'wrong_sense', 'same_pos']

ablation_dirs = {}
ablation_train_files = {}
ablation_val_files   = {}

for strategy in ABLATION_STRATEGIES:
    out_dir = os.path.join(DRIVE_DATASET_ROOT, f'contrastive/youtube_ablation_{strategy}')
    ablation_dirs[strategy]       = out_dir
    ablation_train_files[strategy] = os.path.join(out_dir, 'contrastive_train.csv')
    ablation_val_files[strategy]   = os.path.join(out_dir, 'contrastive_val.csv')

    print(f'\nBuilding ablation dataset: strategy={strategy}')
    %cd /content/concept_aware_training
    !python build_contrastive_dataset.py \
      --source both \
      --strategy {strategy} \
      --max_negatives 10 \
      --output_dir {out_dir}

In [ ]:
# Inspect ablation dataset sizes and coverage
print(f"{'Strategy':<15} {'Train rows':>12} {'Val rows':>10}")
print('-' * 40)
for strategy in ABLATION_STRATEGIES:
    train_f = ablation_train_files[strategy]
    val_f   = ablation_val_files[strategy]
    if os.path.exists(train_f) and os.path.exists(val_f):
        n_train = len(pd.read_csv(train_f))
        n_val   = len(pd.read_csv(val_f))
        print(f"{strategy:<15} {n_train:>12} {n_val:>10}")
    else:
        print(f"{strategy:<15}  {'(not built yet)':<20}")

In [ ]:
# Task 8 Step 2: Train one contrastive model per ablation strategy
# Same hyperparameters as Task 4. Only the negative type changes.

ablation_ckpt_dirs = {}

for strategy in ABLATION_STRATEGIES:
    ckpt_dir = os.path.join(OUTPUT_ROOT, f'contrastive_ablation_{strategy}')
    ablation_ckpt_dirs[strategy] = ckpt_dir
    train_f = ablation_train_files[strategy]
    val_f   = ablation_val_files[strategy]

    if not os.path.exists(train_f):
        print(f'Skipping {strategy} — dataset not built.')
        continue

    print(f'\n{"="*60}')
    print(f'Training ablation model: strategy={strategy}')
    print(f'{"="*60}')

    %cd {SCRIPTS_DIR}
    !python run_clm_contrastive.py \
      --model_name_or_path {MODEL_LOCAL_PATH} \
      --train_file {train_f} \
      --validation_file {val_f} \
      --ncp_alpha 0.5 \
      --contrast_beta 1.0 \
      --block_size 128 \
      --torch_dtype bfloat16 \
      --bf16 True \
      --gradient_accumulation_steps 8 \
      --per_device_train_batch_size 2 \
      --per_device_eval_batch_size 2 \
      --auto_find_batch_size True \
      --overwrite_output_dir \
      --do_train \
      --do_eval \
      --output_dir {ckpt_dir}

    print(f'Done: {ckpt_dir}')

In [ ]:
# Task 8 Step 3: Dual eval on all ablation models + baselines

TASK8_RESULTS = '/content/task8_ablation_results.json'

ablation_ckpts = [
    CLM_CKPT,
    SYN_NCP_CKPT,
    CONTRASTIVE_CKPT,      # full contrastive (all strategies)
] + [ablation_ckpt_dirs[s] for s in ABLATION_STRATEGIES if os.path.exists(ablation_ckpt_dirs.get(s, ''))]

ablation_ckpts = [p for p in ablation_ckpts if os.path.exists(p)]
ckpt_args = ' '.join(ablation_ckpts)

print(f'Evaluating {len(ablation_ckpts)} checkpoints ...')
%cd {SCRIPTS_DIR}
!python eval_concept_ppl.py \
  --checkpoints {ckpt_args} \
  --concept_csv {SYN_CONCEPT_VAL} \
  --vanilla_val {VANILLA_VAL} \
  --results_json {TASK8_RESULTS}

In [ ]:
# Task 8 results: ablation comparison
if os.path.exists(TASK8_RESULTS):
    results = json.load(open(TASK8_RESULTS))
    labels = {
        'standard_syn':                   'CLM (baseline)',
        'syn_custom_loss':                 'Syn-NCP (original)',
        'contrastive':                     'Contrastive — ALL strategies',
        'contrastive_ablation_co_hyponym':  'Contrastive — co-hyponym only',
        'contrastive_ablation_wrong_sense': 'Contrastive — wrong-sense only ★',
        'contrastive_ablation_same_pos':    'Contrastive — same-POS only',
    }
    rows = []
    for r in results:
        if 'error' in r:
            continue
        key = os.path.basename(r['checkpoint'].rstrip('/'))
        rows.append({
            'Model':            labels.get(key, key),
            'NTP PPL':          r.get('ntp_ppl'),
            'NTP Acc':          f"{r.get('ntp_acc', 0):.3f}",
            'Concept PPL':      r.get('concept_ppl'),
            'Coverage':         f"{r.get('concept_coverage_pct', 0):.1f}%",
        })
    df8 = pd.DataFrame(rows)
    print('=== Task 8: Hard-Negative Strategy Ablation ===')
    print('★ = PI\'s recommended safer strategy')
    display(df8)
    shutil.copy(TASK8_RESULTS, os.path.join(OUTPUT_ROOT, 'task8_ablation_results.json'))
else:
    print('Results not yet available — run the eval cell above first.')

---
## Master Summary — All Tasks

Collect all results from Tasks 1–8 into a single comparison table for the paper.

In [ ]:
# Master comparison: NTP PPL, Concept PPL, SNLI acc, SPAM acc

def try_load(path):
    return json.load(open(path)) if os.path.exists(path) else []

# Index by checkpoint name
dual_by_ckpt = {os.path.basename(r['checkpoint'].rstrip('/')): r
                for r in try_load(DUAL_RESULTS_SYN) if 'error' not in r}

snli_ft_by_ckpt = {os.path.basename(r['checkpoint'].rstrip('/')): r
                   for r in try_load(SNLI_FINETUNE_JSON) if 'error' not in r}

spam_ft_by_ckpt = {os.path.basename(r['checkpoint'].rstrip('/')): r
                   for r in try_load(SPAM_FINETUNE_JSON) if 'error' not in r}

model_map = {
    'standard_syn':        'Standard CLM',
    'syn_custom_loss':     'Synonym NCP',
    'hyp_custom_loss':     'Hypernym NCP',
    'diff_ncp_syn':        'Differentiable NCP',
    'contrastive':         'Contrastive (syn+hyp)',
    'contrastive_syn_only':'Contrastive (syn-only)',
}

rows = []
for key, label in model_map.items():
    d = dual_by_ckpt.get(key, {})
    s = snli_ft_by_ckpt.get(key, {})
    p = spam_ft_by_ckpt.get(key, {})
    rows.append({
        'Model':          label,
        'NTP PPL':        d.get('ntp_ppl',     '-'),
        'NTP Acc':        f"{d.get('ntp_acc', float('nan')):.3f}" if d else '-',
        'Concept PPL':    d.get('concept_ppl', '-'),
        'SNLI Acc':       f"{s.get('accuracy', float('nan')):.3f}" if s else '-',
        'SPAM Acc':       f"{p.get('accuracy', float('nan')):.3f}" if p else '-',
    })

df_master = pd.DataFrame(rows)
print('=== Master Comparison Table (Tasks 1–8) ===')
display(df_master)

In [ ]:
# Save master table to Drive as CSV
master_csv = os.path.join(OUTPUT_ROOT, 'master_comparison_tasks_1_8.csv')
df_master.to_csv(master_csv, index=False)
print(f'Saved: {master_csv}')

---
## Troubleshooting

| Error | Fix |
|---|---|
| `CUDA out of memory` | Reduce `--per_device_train_batch_size` to 1 or add `--auto_find_batch_size True` |
| `ModuleNotFoundError: nltk` | Run `!pip install nltk -q` and re-import |
| SpamAssassin load error | Ensure `datasets>=2.14` installed; dataset is `talby/spamassassin` config `"text"` |
| Checkpoint path not found | Verify `OUTPUT_ROOT` matches the Drive path used in the main notebook |
| `context_syn` parse error | CSV stores lists as strings -- `ast.literal_eval()` handles this automatically |
| Ablation dataset has 0 negatives | Some WordNet strategies have sparse coverage on informal text -- report the coverage stat |

**Datasets confirmed (2026-06-22)**:  
- SNLI: `stanfordnlp/snli` -- 550k train / 10k val / 10k test, 3-class NLI  
- SPAM: `talby/spamassassin` config `"text"` -- 10.7k total, binary ham/spam, train-only (auto-split 80/20 inside script)  

**Coverage note**: Always report WordNet coverage (% of concept slots with >=1 hard negative) alongside contrastive results.
